# TM-align 结构比较：ESM3 预测 vs 参考目录（bf_structure）

以 **esm3_structures_by_sample** 为查询、以 **参考目录（如 bf_structure）** 为目标：对每个样本内的每个 ESM3 结构与参考目录内**全部** PDB 做 TM-align。Query=ESM3 预测，Target=参考结构。结果写入 `output_base/<sample_id>/comparison_results.csv`。支持断点续跑。

## 1. 初始化环境

In [ ]:
import sys
import os
from pathlib import Path

# 添加 protflow 到路径：先尝试从 cwd 向上查找项目根，否则用环境变量 PROTFLOW_ROOT
project_root = Path(os.environ.get('PROTFLOW_ROOT', Path.cwd()))
if not (project_root / 'src' / 'protflow').exists():
    project_root = Path.cwd()
    while not (project_root / 'src' / 'protflow').exists() and project_root != project_root.parent:
        project_root = project_root.parent
if (project_root / 'src').exists():
    src_dir = str(project_root / 'src')
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)
    print(f"✓ protflow 路径: {src_dir}")
else:
    raise SystemError("未找到 ProtFlow 项目根（需包含 src/protflow）。请在项目根目录打开 notebook，或设置环境变量 PROTFLOW_ROOT")

from protflow.utils.notebook_utils import init_notebook, CORE_PACKAGES

paths = init_notebook('tm_align_by_sample', packages=CORE_PACKAGES)
WORK_DIR = paths['WORK_DIR']
DATA_DIR = paths['DATA_DIR']

## 2. 配置路径与选项

In [ ]:
# ESM3 预测目录（与 ~/esm3run/predicted/esm3_structures_by_sample 一致）
import os
parent_dir = Path(os.path.expanduser("~/esm3run/predicted/esm3_structures_by_sample"))
# 参考目录（如 bf_structure），每个 ESM3 将与此目录内全部 PDB 做 TM-align
reference_dir = Path(os.path.expanduser("~/esm3run/TPS_database/reviewed_results/bf_structure"))
# 结果输出根目录，每个样本写入 <output_base>/<sample_id>/comparison_results.csv
output_base = WORK_DIR / "tm_align_esm3_vs_bf_structure"
# 断点续跑：若上次未跑完，设为 True 可跳过已有 (Query,Target) 对并追加写入
RESUME = False
# 是否为每个样本生成 comparison_plot.png（需 COLLECT_RESULTS=True）
PLOT = False
# 大批量时建议 False，仅写 CSV 不收集到内存，避免卡死
COLLECT_RESULTS = False
# 并行进程数，大批量时建议 4 或更小
NUM_WORKERS = 4
# 每累积多少条成功结果写一次 CSV；设小一点（如 10000）可尽早看到 CSV 增长，首样本 60 万对时「样本进度」会长时间停在 0%
WRITE_BATCH_SIZE = 10000

## 3. 运行比对（后端在 src）

In [ ]:
from protflow.core.structure_comparison import compare_esm3_samples_vs_reference

if not parent_dir.exists():
    print(f"输入目录(ESM3)不存在: {parent_dir}")
    print("请修改上方 parent_dir 为实际的 esm3_structures_by_sample 路径。")
elif not reference_dir.exists():
    print(f"参考目录不存在: {reference_dir}")
    print("请修改上方 reference_dir 为实际的 bf_structure 路径。")
else:
    # 说明：首样本若有 60 万对，「样本进度」会长时间停在 0%，每约 1 万对会打印一行「已比对 x/total ...」；CSV 每累积 WRITE_BATCH_SIZE 条成功结果写一次
    results_by_sample = compare_esm3_samples_vs_reference(
        parent_dir=parent_dir,
        reference_dir=reference_dir,
        output_base=output_base,
        resume=RESUME,
        collect_results=COLLECT_RESULTS,
        num_workers=NUM_WORKERS,
        write_batch_size=WRITE_BATCH_SIZE,
    )
    if PLOT and COLLECT_RESULTS:
        from protflow.core.structure_comparison import plot_comparison_results
        for sample_id, results in results_by_sample.items():
            if not results:
                continue
            plot_path = output_base / sample_id / "comparison_plot.png"
            try:
                plot_comparison_results(results=results, output_path=plot_path, title=sample_id)
            except Exception as e:
                print(f"绘制 {sample_id} 失败: {e}")
    print(f"完成 {len(results_by_sample)} 个样本，结果见 {output_base}")

## 4. 查看结果摘要（可选）

In [ ]:
if globals().get('results_by_sample'):
    for sample_id, results in list(results_by_sample.items())[:5]:
        print(f"  {sample_id}: {len(results)} 条比对（未收集结果时此处为 0，CSV 已写入）")
    if len(results_by_sample) > 5:
        print(f"  ... 共 {len(results_by_sample)} 个样本")
    print(f"  结果目录: {output_base}")